# Week 4 – Predictive Modeling and Optimization in Logistics Systems

**Logistics Data Analyst Internship — Sri Lakshmi**

This notebook uses a simulated/hypothetical dataset created for academic demonstration. Results are illustrative.

## 1. Problem Definition
Predict `Delivery_Time_days` using shipment mode, distance, shipment weight, transportation cost and inventory level. The workflow includes train/test validation, model comparison, 5-fold cross-validation and a cost-constrained shipment-mode optimization scenario.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('Week_4_Logistics_Simulated_Dataset.csv')
print(df.head())
print('Shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
X=df[['Shipment_Mode','Distance_km','Shipment_Weight_kg','Transportation_Cost','Inventory_Level']]
y=df['Delivery_Time_days']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)
preprocessor=ColumnTransformer([('mode',OneHotEncoder(handle_unknown='ignore'),['Shipment_Mode'])],remainder='passthrough')

In [ ]:
models={'Linear Regression':LinearRegression(),
'Decision Tree':DecisionTreeRegressor(max_depth=6,random_state=42),
'Random Forest':RandomForestRegressor(n_estimators=200,max_depth=10,random_state=42)}
pipelines={}; results=[]
for name,model in models.items():
    pipe=Pipeline([('preprocess',preprocessor),('model',model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    pipelines[name]=pipe
    results.append({'Model':name,'MAE':mean_absolute_error(y_test,pred),
                    'RMSE':mean_squared_error(y_test,pred)**0.5,'R2':r2_score(y_test,pred)})
results_df=pd.DataFrame(results).sort_values('RMSE')
results_df

In [ ]:
kf=KFold(n_splits=5,shuffle=True,random_state=42)
cv_results=[]
for name,pipe in pipelines.items():
    s=cross_validate(pipe,X,y,cv=kf,scoring={'mae':'neg_mean_absolute_error','r2':'r2'})
    cv_results.append({'Model':name,'CV_MAE':-s['test_mae'].mean(),'CV_R2':s['test_r2'].mean()})
pd.DataFrame(cv_results)

In [ ]:
selected_model=results_df.iloc[0]['Model']
selected_pipe=pipelines[selected_model]
pred=selected_pipe.predict(X_test)
plt.figure(figsize=(7,5)); plt.scatter(y_test,pred,alpha=.65)
plt.xlabel('Actual Delivery Time (days)'); plt.ylabel('Predicted Delivery Time (days)')
plt.title(f'Actual vs Predicted – {selected_model}'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,5)); plt.bar(results_df['Model'],results_df['RMSE'])
plt.ylabel('RMSE (days)'); plt.title('Model Comparison by RMSE')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

In [ ]:
rf=pipelines['Random Forest']
names=rf.named_steps['preprocess'].get_feature_names_out()
imp=rf.named_steps['model'].feature_importances_
importance_df=pd.DataFrame({'Feature':names,'Importance':imp}).sort_values('Importance',ascending=False)
print(importance_df)
plt.figure(figsize=(8,5)); plt.barh(importance_df['Feature'][::-1],importance_df['Importance'][::-1])
plt.xlabel('Importance'); plt.title('Random Forest Feature Importance'); plt.tight_layout(); plt.show()

## 5. Optimization Strategy
Select a shipment mode that satisfies a transportation-cost budget while minimizing predicted delivery time. This provides a simple route/mode planning heuristic.

In [ ]:
scenario_distance=900; scenario_weight=500; scenario_inventory=400; budget=2500
mode_cost={'Road':1.8,'Air':6.5,'Sea':1.1}
candidates=[]
for m in ['Road','Air','Sea']:
    cost=round(scenario_distance*mode_cost[m]+scenario_weight*.055,2)
    row=pd.DataFrame({'Shipment_Mode':[m],'Distance_km':[scenario_distance],
                      'Shipment_Weight_kg':[scenario_weight],'Transportation_Cost':[cost],
                      'Inventory_Level':[scenario_inventory]})
    candidates.append([m,cost,float(selected_pipe.predict(row)[0])])
optimization_df=pd.DataFrame(candidates,columns=['Mode','Estimated_Cost','Predicted_Delivery_Days'])
print(optimization_df)
feasible=optimization_df[optimization_df.Estimated_Cost<=budget]
print('\nSelected feasible option:',feasible.sort_values('Predicted_Delivery_Days').iloc[0])

## Recommendations
- Flag shipments with high predicted delivery time.
- Compare modes using delivery time and transportation cost together.
- Prioritize alternatives for strict delivery requirements.
- Monitor important shipment variables as more data becomes available.
- Retrain and validate the model periodically with real operational data.

## Conclusion
The workflow demonstrates data simulation, predictive modeling, evaluation, cross-validation and cost-aware logistics optimization. All data and results are simulated for internship demonstration and are not real company performance.